# Hybrid Multi-Echo Complex MP-PCA

기존 Direct Complex MP-PCA에 Multi-Echo GRE 특성을 추가한 실험적 확장 방식이다.

보고서의 heuristic 구조를 재현한다.

- theoretical threshold
- empirical robust threshold
- echo-wise robust noise
- phase coherence
- temporal coherence
- consistency factor
- Rician-style correction
- safety margin
- confidence-weighted patch aggregation


In [2]:
import gc
from pathlib import Path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

from hybrid_multi_echo_mppca import hybrid_multi_echo_mppca


### 환경설정


In [4]:
NOISE_LEVELS = [10, 30, 50]
PATCH_R = 2

EMPIRICAL_FACTOR = 2.0
TEMPORAL_WEIGHT = 0.3
PHASE_WEIGHT = 0.2
CONSISTENCY_WEIGHT = 0.2
SAFETY_MARGIN = 1.1
ECHO_OUTLIER_Z = 2.5

# HUFS 서버 경로
BASE_DIR = Path.home() / "workspace"
DATA_DIR = BASE_DIR / "data"
NOISY_DIR = DATA_DIR / "noisy_data"
OUT_DIR = BASE_DIR / "results" / "hybrid_multi_echo"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR :", BASE_DIR)
print("DATA_DIR :", DATA_DIR)
print("NOISY_DIR:", NOISY_DIR)
print("OUT_DIR  :", OUT_DIR)


### 1. 원본 데이터 불러오기


In [6]:
orig_data = sio.loadmat(
    DATA_DIR / "meas_gre_dir1.mat",
    simplify_cells=True
)

meas_gre = orig_data["meas_gre"]

print("Original shape :", meas_gre.shape)
print("Original dtype :", meas_gre.dtype)
print("Is complex     :", np.iscomplexobj(meas_gre))


Original shape : (256, 224, 176, 6)
Original dtype : complex128
Is complex     : True


### 2. Noise 데이터 불러오기 및 Complex signal 생성


In [8]:
noisy_datasets = {}

for noise_level in NOISE_LEVELS:
    noisy_path = (
        NOISY_DIR /
        f"noisy_meas_gre_dir1_{noise_level}.mat"
    )

    noisy_data = sio.loadmat(
        noisy_path,
        simplify_cells=True
    )

    noisy_real = noisy_data["noisy_real"].astype(np.float32)
    noisy_imag = noisy_data["noisy_imag"].astype(np.float32)
    mask = noisy_data["mask_brain"].astype(bool)

    noisy_complex = (
        noisy_real.astype(np.complex64)
        + 1j * noisy_imag.astype(np.complex64)
    )

    noisy_datasets[noise_level] = {
        "noisy_complex": noisy_complex,
        "mask": mask,
        "metadata": noisy_data
    }

    print("\n" + "=" * 50)
    print(f"Noise level : {noise_level}%")
    print("Complex shape :", noisy_complex.shape)
    print("Complex dtype :", noisy_complex.dtype)
    print("=" * 50)



Noise level : 10%
Complex shape : (256, 224, 176, 6)
Complex dtype : complex64

Noise level : 30%
Complex shape : (256, 224, 176, 6)
Complex dtype : complex64

Noise level : 50%
Complex shape : (256, 224, 176, 6)
Complex dtype : complex64


### 3. Hybrid Multi-Echo Complex MP-PCA 실행


In [ ]:
denoised_data = {}

for noise_level in NOISE_LEVELS:
    noisy_complex = noisy_datasets[noise_level]["noisy_complex"]
    mask = noisy_datasets[noise_level]["mask"]

    print("\n" + "=" * 60)
    print(f"Noise level : {noise_level}%")
    print("Hybrid Multi-Echo Complex MP-PCA")
    print("=" * 60)

    deno_complex, maps = hybrid_multi_echo_mppca(
        noisy_complex,
        mask=mask,
        patch_radius=PATCH_R,
        empirical_factor=EMPIRICAL_FACTOR,
        temporal_weight=TEMPORAL_WEIGHT,
        phase_weight=PHASE_WEIGHT,
        consistency_weight=CONSISTENCY_WEIGHT,
        safety_margin=SAFETY_MARGIN,
        echo_outlier_z=ECHO_OUTLIER_Z,
        use_rician_correction=True,
        return_maps=True
    )

    denoised_data[noise_level] = {
        "deno_complex": deno_complex,
        "maps": maps
    }

    print("Output shape :", deno_complex.shape)
    print("Output dtype :", deno_complex.dtype)
    print("Is complex   :", np.iscomplexobj(deno_complex))

    inside = mask
    print(
        "Mean phase coherence    :",
        float(np.mean(maps["phase_coherence"][inside]))
    )
    print(
        "Mean temporal coherence :",
        float(np.mean(maps["temporal_coherence"][inside]))
    )
    print(
        "Mean retained comps     :",
        float(np.mean(maps["nsignal_map"][inside]))
    )



Noise level : 10%
Hybrid Multi-Echo Complex MP-PCA


### 4. 결과 저장


In [ ]:
for noise_level in NOISE_LEVELS:
    meta = noisy_datasets[noise_level]["metadata"]
    result = denoised_data[noise_level]

    output_path = (
        OUT_DIR /
        f"denoised_hybrid_multi_echo_{noise_level}_r{PATCH_R}.mat"
    )

    save_data = {
        "denoised_complex": result["deno_complex"],
        "phase_coherence": result["maps"]["phase_coherence"],
        "temporal_coherence": result["maps"]["temporal_coherence"],
        "threshold_map": result["maps"]["threshold_map"],
        "confidence_map": result["maps"]["confidence_map"],
        "nsignal_map": result["maps"]["nsignal_map"],
        "robust_noise_map": result["maps"]["robust_noise_map"],
        "mask_brain": meta["mask_brain"],
        "noise_level": noise_level,
        "patch_radius": PATCH_R,
        "empirical_factor": EMPIRICAL_FACTOR,
        "temporal_weight": TEMPORAL_WEIGHT,
        "phase_weight": PHASE_WEIGHT,
        "consistency_weight": CONSISTENCY_WEIGHT,
        "safety_margin": SAFETY_MARGIN,
        "echo_outlier_z": ECHO_OUTLIER_Z,
    }

    for key in ["te_gre", "delta_TE", "voxel_size_gre", "B0_dir", "CF"]:
        if key in meta:
            save_data[key] = meta[key]

    sio.savemat(
        output_path,
        save_data
    )

    print("저장 완료 →", output_path)


### 5. Magnitude 시각화


In [ ]:
orig_mag = np.abs(meas_gre).astype(np.float32)

echo_idx = 0
slice_idx = meas_gre.shape[2] // 2

for noise_level in NOISE_LEVELS:
    noisy_complex = noisy_datasets[noise_level]["noisy_complex"]
    deno_complex = denoised_data[noise_level]["deno_complex"]
    mask = noisy_datasets[noise_level]["mask"]

    noisy_mag = np.abs(noisy_complex).astype(np.float32)
    deno_mag = np.abs(deno_complex).astype(np.float32)

    orig_2d = orig_mag[:, :, slice_idx, echo_idx]
    noisy_2d = noisy_mag[:, :, slice_idx, echo_idx]
    deno_2d = deno_mag[:, :, slice_idx, echo_idx]
    mask_2d = mask[:, :, slice_idx]

    diff_2d = deno_2d - orig_2d

    vmin, vmax = np.percentile(
        orig_mag[mask],
        (1, 99)
    )

    dv = np.percentile(
        np.abs(diff_2d[mask_2d]),
        99
    )

    fig, ax = plt.subplots(
        1,
        4,
        figsize=(18, 4)
    )

    ax[0].imshow(
        orig_2d.T,
        cmap="gray",
        origin="lower",
        vmin=vmin,
        vmax=vmax
    )
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(
        noisy_2d.T,
        cmap="gray",
        origin="lower",
        vmin=vmin,
        vmax=vmax
    )
    ax[1].set_title(f"Noisy ({noise_level}%)")
    ax[1].axis("off")

    ax[2].imshow(
        deno_2d.T,
        cmap="gray",
        origin="lower",
        vmin=vmin,
        vmax=vmax
    )
    ax[2].set_title("Hybrid Multi-Echo")
    ax[2].axis("off")

    ax[3].imshow(
        diff_2d.T,
        cmap="bwr",
        origin="lower",
        vmin=-dv,
        vmax=dv
    )
    ax[3].set_title("Difference")
    ax[3].axis("off")

    fig.suptitle(
        f"Hybrid Multi-Echo Complex MP-PCA | Noise {noise_level}% | "
        f"Echo {echo_idx + 1} | Slice {slice_idx}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


### 6. Coherence / Threshold / Component map 확인


In [ ]:
slice_idx = meas_gre.shape[2] // 2

for noise_level in NOISE_LEVELS:
    maps = denoised_data[noise_level]["maps"]

    fig, ax = plt.subplots(
        1,
        4,
        figsize=(18, 4)
    )

    im0 = ax[0].imshow(
        maps["phase_coherence"][:, :, slice_idx].T,
        origin="lower",
        vmin=0,
        vmax=1
    )
    ax[0].set_title("Phase coherence")
    ax[0].axis("off")
    plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

    im1 = ax[1].imshow(
        maps["temporal_coherence"][:, :, slice_idx].T,
        origin="lower",
        vmin=0,
        vmax=1
    )
    ax[1].set_title("Temporal coherence")
    ax[1].axis("off")
    plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

    im2 = ax[2].imshow(
        maps["threshold_map"][:, :, slice_idx].T,
        origin="lower"
    )
    ax[2].set_title("Adaptive threshold")
    ax[2].axis("off")
    plt.colorbar(im2, ax=ax[2], fraction=0.046, pad=0.04)

    im3 = ax[3].imshow(
        maps["nsignal_map"][:, :, slice_idx].T,
        origin="lower",
        vmin=0,
        vmax=meas_gre.shape[-1]
    )
    ax[3].set_title("Retained components")
    ax[3].axis("off")
    plt.colorbar(im3, ax=ax[3], fraction=0.046, pad=0.04)

    fig.suptitle(
        f"Hybrid diagnostic maps | Noise {noise_level}%",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


### 7. 메모리 정리


In [ ]:
del noisy_datasets
del denoised_data

gc.collect()

print("메모리 정리 완료")
